**Boolean logic gates as matrices.**
Classical logic gates can be represented as matrices that act on **one-hot column vectors**.
Each logical value is encoded as a vector in which exactly one entry is 1 and the rest are 0;
the *position* of the 1 encodes the state:

$$\mathbf{F} = \begin{bmatrix}1\\0\end{bmatrix}\quad(\text{index 0 = False})
\qquad
\mathbf{T} = \begin{bmatrix}0\\1\end{bmatrix}\quad(\text{index 1 = True})$$

This encoding is identical to the quantum computational basis:
$|0\rangle$ for False and $|1\rangle$ for True.
A logical gate is then a **matrix** that maps one-hot input vectors to one-hot output vectors,
and **composing gates is matrix multiplication**.

In [ ]:
"""gate_matrices.ipynb"""

# Cell 01 - Imports and True & False as one-hot column vectors

import numpy as np
from IPython.display import Math, display

from qis101_utils import as_latex

f = np.array([[1], [0]])  # False = |0>
t = np.array([[0], [1]])  # True  = |1>

display(as_latex(f, prefix=r"\mathbf{F}=0="))
display(as_latex(t, prefix=r"\mathbf{T}=1="))

**NOT gate.**
The NOT gate swaps the two entries of the input vector.
Its matrix is the $2\times 2$ permutation matrix:

$$\mathbf{NOT} = \begin{bmatrix}0 & 1 \\ 1 & 0\end{bmatrix}$$

This is also the Pauli-$X$ gate in quantum computing.

In [ ]:
# Cell 02 - Helper functions and NOT gate


def expected_name(vec: np.ndarray) -> str:
    """Return 'T' or 'F' based on which one-hot vector vec matches."""
    if np.allclose(vec, t):
        return "T"
    if np.allclose(vec, f):
        return "F"
    return "?"


def verify(gate_name: str, inputs_name: str, result: np.ndarray, expected: np.ndarray):
    """Display a checkmark or cross confirming gate output matches expected vector."""
    symbol = r"\checkmark" if np.allclose(result, expected) else r"\times"
    display(
        Math(
            rf"\mathbf{{{gate_name}\;({inputs_name})}} "
            rf"= \mathbf{{{expected_name(expected)}}} \;\rightarrow\; {symbol}"
        )
    )


g_not = np.array([[0, 1], [1, 0]])

display(as_latex(g_not, prefix=r"\mathbf{NOT}="))

verify("NOT", "F", g_not @ f, t)
verify("NOT", "T", g_not @ t, f)

**Two-input states via the Kronecker (tensor) product.**
For two-input gates we need a 4-element vector representing all combinations of two logical values.
The **Kronecker product** $\mathbf{A} \otimes \mathbf{B}$ constructs this automatically:
the resulting vector has a 1 at position $2a + b$ where $a, b \in \{0,1\}$,
giving the four basis states $(0,0)=0$, $(0,1)=1$, $(1,0)=2$, $(1,1)=3$.

In [ ]:
# Cell 03 - Two-input basis states via Kronecker product

f_f = np.kron(f, f)
f_t = np.kron(f, t)
t_f = np.kron(t, f)
t_t = np.kron(t, t)

display(as_latex(f_f, prefix=r"\mathbf{F\;F}=(0,0)=0="))
display(as_latex(f_t, prefix=r"\mathbf{F\;T}=(0,1)=1="))
display(as_latex(t_f, prefix=r"\mathbf{T\;F}=(1,0)=2="))
display(as_latex(t_t, prefix=r"\mathbf{T\;T}=(1,1)=3="))

**AND gate.**
AND takes a 4-element input vector and produces a 2-element output vector,
so its matrix has shape $(2\times 4)$.
The output is True only when both inputs are True (the last basis state):

$$\mathbf{AND} = \begin{bmatrix}1&1&1&0\\0&0&0&1\end{bmatrix}$$

In [ ]:
# Cell 04 - AND gate (2x4 matrix: 2 inputs -> 1 output)

g_and = np.array([[1, 1, 1, 0], [0, 0, 0, 1]])

display(as_latex(g_and, prefix=r"\mathbf{AND}="))

verify("AND", "F,F", g_and @ f_f, f)
verify("AND", "F,T", g_and @ f_t, f)
verify("AND", "T,F", g_and @ t_f, f)
verify("AND", "T,T", g_and @ t_t, t)

**OR gate.**
OR outputs True when at least one input is True (all basis states except $(0,0)$):

$$\mathbf{OR} = \begin{bmatrix}1&0&0&0\\0&1&1&1\end{bmatrix}$$

In [ ]:
# Cell 05 - OR gate (2x4 matrix)

g_or = np.array([[1, 0, 0, 0], [0, 1, 1, 1]])

display(as_latex(g_or, prefix=r"\mathbf{OR}="))

verify("OR", "F,F", g_or @ f_f, f)
verify("OR", "F,T", g_or @ f_t, t)
verify("OR", "T,F", g_or @ t_f, t)
verify("OR", "T,T", g_or @ t_t, t)

**Composing gates: NAND and NOR.**
AND maps a 4-element input to a 2-element output: shape $(2\times 4)$.
NOT maps a 2-element vector to a 2-element vector: shape $(2\times 2)$.
Because AND's output dimension matches NOT's input dimension they can be chained directly:

$$\mathbf{NAND} = \mathbf{NOT}\cdot\mathbf{AND}
\quad\Rightarrow\quad
(2\times 2)\cdot(2\times 4) = (2\times 4)$$

This is not a coincidence - it is exactly why the matrix representation is useful.
**Composing gates is matrix multiplication**, and the algebra of gates is the algebra of matrices.

In [ ]:
# Cell 06 - NAND and NOR via matrix composition

g_nand = g_not @ g_and
g_nor = g_not @ g_or

display(as_latex(g_nand, prefix=r"\mathbf{NAND}="))

verify("NAND", "F,F", g_nand @ f_f, t)
verify("NAND", "F,T", g_nand @ f_t, t)
verify("NAND", "T,F", g_nand @ t_f, t)
verify("NAND", "T,T", g_nand @ t_t, f)

display(as_latex(g_nor, prefix=r"\mathbf{NOR}="))

verify("NOR", "F,F", g_nor @ f_f, t)
verify("NOR", "F,T", g_nor @ f_t, f)
verify("NOR", "T,F", g_nor @ t_f, f)
verify("NOR", "T,T", g_nor @ t_t, f)

**XOR - the gate that requires its own matrix.**
XOR (exclusive-or) outputs True when **exactly one** input is True.
Unlike NAND and NOR, XOR cannot be expressed as a single matrix product of NOT, AND, and OR -
it requires its own $(2\times 4)$ matrix:

$$\mathbf{XOR} = \begin{bmatrix}1&0&0&1\\0&1&1&0\end{bmatrix}$$

XOR is the direct classical predecessor of the quantum **Controlled-NOT (CNOT)** gate,
which you will encounter in later sessions.

In [ ]:
# Cell 07 - XOR gate (requires its own matrix; not composable from NOT/AND/OR)

g_xor = np.array([[1, 0, 0, 1], [0, 1, 1, 0]])

display(as_latex(g_xor, prefix=r"\mathbf{XOR}="))

verify("XOR", "F,F", g_xor @ f_f, f)
verify("XOR", "F,T", g_xor @ f_t, t)
verify("XOR", "T,F", g_xor @ t_f, t)
verify("XOR", "T,T", g_xor @ t_t, f)